In [1]:
import pandas as pd
import numpy as np
import os
import time
import pickle
from matplotlib import pyplot as plt
import seaborn as sns

os.chdir(os.getcwd())
os.getcwd()

'/data/gpfs/projects/punim2121/C-Path/outcome_prediction/outcome_prediction'

<hr>

#### DROP MGIT MEASUREMENTS THAT HAVE BEEN PROVED TO BE FALSE POSITIVES. 
#### THIS INFORMATION IS ONLY AVAILABLE IN THE TB-1021 & AND MAYBE TB-1018 STUDY

In [2]:
# load necessary dataframes and study+pat_id data
pat_id_df = pd.read_csv('../data/patients_in_analysis.csv.gz', index_col=0)
#xo=pd.read_csv('../../C-Path_data/fullExportDb-1025-Member-CSV/xo.csv', low_memory=False)
studies = pat_id_df['STUDYID'].unique()
pat_ids = pat_id_df['USUBJID'].values.tolist()
mb=pd.read_csv('../../C-Path_data/fullExportDb-1025-Member-CSV/mb_with_results_days_arms.csv', low_memory=False)
#mb=mb[mb['USUBJID'].isin(pat_ids)].dropna(how='all',axis=1)

<hr>

##### **DROP FALSE POSITIVES TTP MEASUREMENTS OF TB-1021 & TB-1018 STUDIES FROM THE MB DATABASE**
* ##### THIS IS MEASURED BY A BLOOD AGAR CULTURE VALIDATION AFTER THE MGIT SIGNALS POSITIVE, 
* ##### IF THE BLOOD-AGAR CULTURE GROWS A NON-MTB BACTERIUM OR FUNGHI, THE TTP WAS FALSE POSITIVE

In [3]:

## Create unique sample reference ID by concatenating Patient ID + Microbiological Reference ID of the sample
mb['SAMPLE_REFID'] = np.nan
mb.loc[~mb['MBREFID'].isna(), 'SAMPLE_REFID'] = mb.loc[~mb['MBREFID'].isna(), 'STUDYID'].astype(str) + \
    '_'+mb.loc[~mb['MBREFID'].isna(), 'MBREFID'].astype(str)

mb_=mb[mb['USUBJID'].isin(pat_id_df['USUBJID'])]
mb_.groupby(['STUDYID','MBTESTCD','MEDIATYP'],dropna=False).apply(lambda x: (x['MBTSTDTL'].value_counts()))#.reset_index()

/tmp/ipykernel_211693/3515180302.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['TB-1021_200335' 'TB-1021_200174' 'TB-1021_200156' ... 'TB-1018_NIX-1930'
 'TB-1018_P1163436' 'TB-1018_DNIX-0350']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mb.loc[~mb['MBREFID'].isna(), 'SAMPLE_REFID'] = mb.loc[~mb['MBREFID'].isna(), 'STUDYID'].astype(str) + \
/tmp/ipykernel_211693/3515180302.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mb_.groupby(['STUDYID','MBTESTCD','MEDIATYP'],dropna=False).apply(lambda x: (x['MBTSTDTL'].value_counts()))#.reset_index()


STUDYID  MBTESTCD  MEDIATYP                  MBTSTDTL                            
TB-1018  AFB       NaN                       Identification                            787
                                             Categorical Count                         130
         MTB       NaN                       Culture Growth                           3629
                                             Time to Detection                         520
         MTBCMPLX  NaN                       MPT64 Antigen Test                        332
                                             Identification                            139
                                             Minimum Cycle Threshold of Detection       10
         NONMTB    BLOOD AGAR                Culture Growth                            631
TB-1020  AFB       NaN                       Categorical Count                       10772
         MTB       LOWENSTEIN JENSEN MEDIUM  Culture Growth                           3705
        

# SUMMARIZE MB TEST RESULT FOR EACH VISIT

## TB-1022

In [4]:

'''
def get_mode_of_result(x):
    if len(x['RESULT'].mode())==1:
        return x['RESULT'].mode()[0]
        
    if len(x['RESULT'].mode())==2:
        return 'positive'

    if len(x['RESULT'].mode())>2:
        raise ValueError('Invalid mb result! (not "negative" or "positive")')   
'''
study_name='TB-1022'
mb_subset=mb_[mb_['STUDYID']==study_name].dropna(how='all',axis=1)


## EXTRACT VISITS OF PATIENTS WHERE THE DAY INFORMATION IS PROBABLY INCORRECT (DAY OF SAMPLE IS <-20, IN THE RANGE OF (-30,-600)
## IF THERE ARE DATA PRESENT FROM THE SAME VISIT AND THE DAY OF SAMPLE COLLECTION SEEMS MORE PLAUSIBLE, CORRECT THESE LARGE NEGATIVE DAYS WITH THEM
pats_with_incorr_days=mb_subset[(mb_subset['MBDY_estimated']<-20)]['USUBJID'].unique()

visits_with_incorr_day=mb_subset[(mb_subset['USUBJID'].isin(pats_with_incorr_days))&((mb_subset['MBDY_estimated']<-20))][['USUBJID','VISIT']]#.unique()

for pat_visit,pat_df in visits_with_incorr_day.groupby(['USUBJID','VISIT']):
    pat=pat_visit[0]
    visit=pat_visit[1]

    ## DAYS OF THE UNSCHEDULED VISIT CAN'T BE DEDUCED UNFORTUNATELY
    if 'UNSCHEDULED'not in visit:
        #print(pat,visit)
        
        corr_day=(mb_subset[(mb_subset['USUBJID']==pat)\
                        &(mb_subset['VISIT']==visit)\
                        &(mb_subset['MBDY_estimated']>-20)]['MBDY_estimated'].min())
        mb_subset.loc[(mb_subset['USUBJID']==pat)\
                        &(mb_subset['VISIT']==visit)\
                        &(mb_subset['MBDY_estimated']<-20),'MBDY_estimated']=corr_day



## CONVERT ZN-SMEAR RESULTS TO THE CDC QUANTIFICATION LEVELS (based on: remox-laboratory-manual.pdf page 15, 
#. also used in publication: https://www.nature.com/articles/s41591-018-0224-2#Sec9
tb_22_21_ZN_conversion_table={'NEGATIVE':'NEGATIVE',
                                'SCANTY':'1+',
                                 '1+':'2+',
                                 '2+':'3+',
                                 '3+':'4+'}

## Create new column with converting ZN-smear results to ordinal levels
tb_22_21_ZN_ordinal_conversion_table={'NEGATIVE':0,
                                        '1+':1,
                                        '2+':2,
                                        '3+':3,
                                        '4+':4}

mb_subset.loc[mb_subset['MBTESTCD']=='AFB','STD_CAT_RESULT']=mb_subset.loc[mb_subset['MBTESTCD']=='AFB','STD_CAT_RESULT'].map(tb_22_21_ZN_conversion_table)
mb_subset['STD_CAT_ORDINAL_RESULT']=np.nan
mb_subset.loc[mb_subset['MBTESTCD']=='AFB','STD_CAT_ORDINAL_RESULT'] = mb_subset.loc[mb_subset['MBTESTCD']=='AFB','STD_CAT_RESULT'].map(tb_22_21_ZN_ordinal_conversion_table)



## Get mode of binary results
def get_mode_of_result(x):
    if len(x['RESULT'].mode())==1:
        res = x['RESULT'].mode()[0]
        
        if res=='positive':
            res_cat=x['STD_CAT_ORDINAL_RESULT'].max()
        if res=='negative':
            res_cat=x['STD_CAT_ORDINAL_RESULT'].min()
        return pd.Series([res,res_cat])
        
    if len(x['RESULT'].mode())==2:
        return pd.Series(['positive',x['STD_CAT_ORDINAL_RESULT'].max()])

    if len(x['RESULT'].mode())>2:
        raise ValueError('Invalid mb result! (not "negative" or "positive")')  

a=mb_subset.groupby(['USUBJID','MBDY_estimated', 'STUDYID','MBTESTCD','MBMETHOD','MBTSTDTL','MEDIATYP'],dropna=False).apply(get_mode_of_result)

tb_22_std_res=a.reset_index()
tb_22_std_res=tb_22_std_res.rename(columns={0:'STD_RESULT',1:'STD_CAT_ORDINAL_RESULT'})
tb_22_std_res['STD_CAT_RESULT'] = tb_22_std_res['STD_CAT_ORDINAL_RESULT'].map(dict(zip(tb_22_21_ZN_ordinal_conversion_table.values(),tb_22_21_ZN_ordinal_conversion_table.keys())))
tb_22_std_res['STD_TEST']=np.nan


tb_22_std_res.loc[tb_22_std_res['MBTESTCD']=='AFB','STD_TEST']='ZN-smear'
tb_22_std_res.loc[tb_22_std_res['MBTESTCD']=='MTB','STD_TEST']='LJ-culture'

## Add column CULTURE_STATUS ==> 
## copy of LJ culture results, as there were no instructions on how to define culture status during treatment phase (unlike with tb-1021, see below)
## ==> only define for end of treatment timepoint and after (see TB-1022 protocol, appendix 10. (page 192/251)
## ==> PROBLEM: TREATMENT CURE/FAILURE IS DEFINED ON PAGE 28/251:
#.              CURE: "is defined as the presence of two negative cultures in two sputum collections taken at least one day apart at the end of treatment"
#               ===> MAJORITY PATIENTS HAVE ONLY ONE CULTURE RESULT AT THE END OF THERAPY VISIT, OR LATER, DATA IS MISSING ??
#.              ===> EXACT RECONSTRUCTION OF FAILURE AND CURE IS NOT POSSIBLE BASED ON THE LJ-CULTURE DATA ON HAND

## ==> Consider the LJ culture results as culture status
tb_22_std_res['CULTURE_STATUS']=np.nan
tb_22_std_res.loc[tb_22_std_res['MBTESTCD']=='MTB','CULTURE_STATUS']=tb_22_std_res.loc[tb_22_std_res['MBTESTCD']=='MTB','STD_RESULT'].values





/tmp/ipykernel_211693/2435939164.py:77: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  a=mb_subset.groupby(['USUBJID','MBDY_estimated', 'STUDYID','MBTESTCD','MBMETHOD','MBTSTDTL','MEDIATYP'],dropna=False).apply(get_mode_of_result)
/tmp/ipykernel_211693/2435939164.py:85: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'ZN-smear' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  tb_22_std_res.loc[tb_22_std_res['MBTESTCD']=='AFB','STD_TEST']='ZN-smear'
/tmp/ipykernel_211693/2435939164.py:98: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a 

In [6]:
mb_subset.loc[mb_subset['MBTESTCD']=='AFB',:].dropna(how='all',axis=1)

,STUDYID,DOMAIN,USUBJID,MBSEQ,MBSPID,MBTESTCD,MBTEST,MBTSTDTL,MBORRES,MBSTRESC,MBSPEC,VISITNUM,VISIT,MBDY,RESULT,STD_CAT_RESULT,MBDY_estimated,ARM,STD_CAT_ORDINAL_RESULT
216512,TB-1022,MB,TB-1022/11023,2,2,AFB,Acid-Fast Bacilli,Categorical Count,3+,3+,SPUTUM,1.00,VISIT 1: SCREENING,-2.0,positive,4+,-2.0,Gati-arm regimen (4 month regimen),4.0
216708,TB-1022,MB,TB-1022/11024,2,2,AFB,Acid-Fast Bacilli,Categorical Count,2+,2+,SPUTUM,1.00,VISIT 1: SCREENING,-2.0,positive,3+,-2.0,Control-arm regimen (6 month regimen),3.0
216954,TB-1022,MB,TB-1022/11025,2,2,AFB,Acid-Fast Bacilli,Categorical Count,2+,2+,SPUTUM,1.00,VISIT 1: SCREENING,-3.0,positive,3+,-3.0,Gati-arm regimen (4 month regimen),3.0
217062,TB-1022,MB,TB-1022/11026,2,2,AFB,Acid-Fast Bacilli,Categorical Count,3+,3+,SPUTUM,1.00,VISIT 1: SCREENING,-3.0,positive,4+,-3.0,Control-arm regimen (6 month regimen),4.0
217142,TB-1022,MB,TB-1022/11027,2,2,AFB,Acid-Fast Bacilli,Categorical Count,3+,3+,SPUTUM,1.00,VISIT 1: SCREENING,-1.0,positive,4+,-1.0,Control-arm regimen (6 month regimen),4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
312870,TB-1022,MB,TB-1022/52367,34,2,AFB,Acid-Fast Bacilli,Categorical Count,NEGATIVE,NEGATIVE,SPUTUM,99.03,UNSCHEDULED 03,821.0,negative,NEGATIVE,821.0,NaN,0.0
312871,TB-1022,MB,TB-1022/51404,35,1,AFB,Acid-Fast Bacilli,Categorical Count,NEGATIVE,NEGATIVE,SPUTUM,18.00,VISIT 18: WEEK 112/120,783.0,negative,NEGATIVE,783.0,NaN,0.0
312872,TB-1022,MB,TB-1022/52367,35,1,AFB,Acid-Fast Bacilli,Categorical Count,NEGATIVE,NEGATIVE,SPUTUM,99.04,UNSCHEDULED 04,961.0,negative,NEGATIVE,961.0,NaN,0.0
312873,TB-1022,MB,TB-1022/51404,36,2,AFB,Acid-Fast Bacilli,Categorical Count,NEGATIVE,NEGATIVE,SPUTUM,18.00,VISIT 18: WEEK 112/120,784.0,negative,NEGATIVE,784.0,NaN,0.0


In [ ]:
mb_subset[mb_subset['MBTESTCD']=='AFB']['MBSTRESC'].value_counts()

In [ ]:
tb_22_std_res.loc[(tb_22_std_res['STD_RESULT']=='positive')&\
                  (tb_22_std_res['MBTESTCD']!='AFB'),'STD_CAT_ORDINAL_RESULT'].value_counts(dropna=False)

## TB-1021

In [11]:
mb_subset_clean.dropna(how='all',axis=1)

mb_subset_clean.loc[mb_subset_clean['MBTESTCD']=='AFB',:].dropna(how='all',axis=1)

,STUDYID,DOMAIN,USUBJID,MBSEQ,MBREFID,MBTESTCD,MBTEST,MBTSTDTL,MBORRES,MBSTRESC,...,EPOCH,MBDTC,MBDY,MBEVINTX,RESULT,STD_CAT_RESULT,MBDY_estimated,ARM,SAMPLE_REFID,STD_RESULT
5703,TB-1021,MB,TB-1021/1029832,9,300967,AFB,Acid-Fast Bacilli,Categorical Count,++++,4+,...,INTENSIVE PHASE,-----T06:00,1.0,EARLY MORNING COLLECTION,positive,4+,1.0,NaN,TB-1021_300967,positive
154286,TB-1021,MB,TB-1021/2643925,93,306078,AFB,Acid-Fast Bacilli,Categorical Count,NO AFB SEEN,NEGATIVE,...,FOLLOW-UP,-----T11:15,448.0,SPOT COLLECTION,negative,NEGATIVE,448.0,NaN,TB-1021_306078,negative
154574,TB-1021,MB,TB-1021/2688081,93,0507584,AFB,Acid-Fast Bacilli,Categorical Count,NO AFB SEEN,NEGATIVE,...,FOLLOW-UP,-----T11:15,550.0,SPOT COLLECTION,negative,NEGATIVE,550.0,NaN,TB-1021_0507584,negative
154802,TB-1021,MB,TB-1021/2086276,94,305817,AFB,Acid-Fast Bacilli,Categorical Count,NO AFB SEEN,NEGATIVE,...,FOLLOW-UP,-----T11:15,469.0,SPOT COLLECTION,negative,NEGATIVE,469.0,NaN,TB-1021_305817,negative
154930,TB-1021,MB,TB-1021/1358618,95,513525,AFB,Acid-Fast Bacilli,Categorical Count,NO AFB SEEN,NEGATIVE,...,FOLLOW-UP,-----T11:15,352.0,SPOT COLLECTION,negative,NEGATIVE,352.0,NaN,TB-1021_513525,negative
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
228007,TB-1021,MB,TB-1021/2398493,8,1300154,AFB,Acid-Fast Bacilli,Categorical Count,++++,4+,...,SCREENING,-----T23:00,-1.0,SPOT COLLECTION,positive,4+,-1.0,NaN,TB-1021_1300154,positive
228013,TB-1021,MB,TB-1021/1515564,1,700372,AFB,Acid-Fast Bacilli,Categorical Count,++++,4+,...,SCREENING,NaN,-1.0,SPOT COLLECTION,positive,4+,-1.0,NaN,TB-1021_700372,positive
228027,TB-1021,MB,TB-1021/1001933,17,300015,AFB,Acid-Fast Bacilli,Categorical Count,++++,4+,...,INTENSIVE PHASE,NaN,8.0,EARLY MORNING COLLECTION,positive,4+,8.0,NaN,TB-1021_300015,positive
228033,TB-1021,MB,TB-1021/1013413,24,400072,AFB,Acid-Fast Bacilli,Categorical Count,++,2+,...,INTENSIVE PHASE,NaN,14.0,EARLY MORNING COLLECTION,positive,2+,14.0,NaN,TB-1021_400072,positive


In [7]:
study_name='TB-1021'
mb_subset=mb[mb['STUDYID']==study_name].dropna(how='all',axis=1)
mb_subset[['STD_RESULT']]=mb_subset[['RESULT']].values
print('Num of initial samples',len(mb_subset))

##======== IDENTIFY CONTAMINATED POSITIVE MGIT SAMPLES ===============
#  Based on: https://www.ucl.ac.uk/infection-immunity/sites/infection_immunity/files/remox-laboratory-manual.pdf, page 10-25

## 1. Positive MGIT results are getting tested for contamination by inoculating blood agar with a small sample 
#     from the MGIT tube 
#  2. If the blood agar ('MBTESTCD'=='NONMTB') is negative ('MBSTRESC'=='NEGATIVE (MGIT RESULT VALID)'), 
#     the MGIT is considered not contaminated
not_cont_pos_ttp_sample_ids=mb_subset.loc[(mb_subset['MBTESTCD']=='NONMTB')&\
                                          (mb_subset['MBSTRESC']=='NEGATIVE (MGIT RESULT VALID)'),'SAMPLE_REFID'].dropna().unique()

## 3. Blood agar from contaminated positive MGIT results are NOT REPORTED 
#     ==> we have to infer them by subtracting the not-contaminated MGIT sample IDs from all MGIT sample IDs
all_pos_ttp_sample_ids=mb_subset.loc[(mb_subset['MBTSTDTL']=='Time to Detection')&\
                                     (~mb_subset['MBSTRESN'].isna()),'SAMPLE_REFID'].unique()   

contam_ttp_sample_ids=list(set(all_pos_ttp_sample_ids) - set(not_cont_pos_ttp_sample_ids))
print(len(mb_subset))


##======= DROP CONTAMINATED POSITIVE MGIT SAMPLES + 
#         LINKED ZN-SMEAR RESULTS PERFORMED ON CONT.MGIT + 
#         BLOOD AGAR MEASUREMENTS OF VALIDATION ===============
cont_ttp_idx=mb_subset.loc[(mb_subset['SAMPLE_REFID'].isin(contam_ttp_sample_ids))
                           &(mb_subset['MBGRPID'].str.contains('MGIT'))].index.tolist()

blood_agar_idx=mb_subset.loc[(mb['MBTESTCD']=='NONMTB')].index.tolist()   
mb_subset_clean=mb_subset.drop(index=cont_ttp_idx + blood_agar_idx)
print('Num of samples after dropping contaminated MGIT + blood agar samples',len(mb_subset_clean))



##======== IDENTIFY FALSE POSITIVE MGIT SAMPLES ===============
#  Based on: https://www.ucl.ac.uk/infection-immunity/sites/infection_immunity/files/remox-laboratory-manual.pdf, page 10-25
#            and https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5658986/ , microbiology section
#  1. All cultures that are positive (LJ or MGIT) are getting a ZN-smear from an isolate taken from that culture
#  2. If MGIT is positive but the validation ZN-smear is negative, the MGIT is considered as false positive
#  
## Get samples where the validation ZN-smears of MGIT tests were negative
mgit_samples_with_neg_afb=mb_subset_clean.loc[(mb_subset_clean['MBTESTCD']=='AFB')&\
                      (mb_subset_clean['MBGRPID'].str.contains('MGIT'))&
                      (mb_subset_clean['RESULT']=='negative'),'SAMPLE_REFID'].tolist()

## Get samples where the validation ZN-smears of LJ tests were negative
lj_samples_with_neg_afb=mb_subset_clean.loc[(mb_subset_clean['MBTESTCD']=='AFB')&\
                      (mb_subset_clean['MBGRPID'].str.contains('LJ'))&
                      (mb_subset_clean['RESULT']=='negative'),'SAMPLE_REFID'].tolist()                      

## Extract the results of the matching culture - validation ZN smears
a=(mb_subset_clean[mb_subset_clean['SAMPLE_REFID'].isin(mgit_samples_with_neg_afb)].groupby(['SAMPLE_REFID','MBGRPID','MBTESTCD','MBTSTDTL'],dropna=True).apply(lambda x:x['RESULT'])).reset_index()
a_=(mb_subset_clean[mb_subset_clean['SAMPLE_REFID'].isin(lj_samples_with_neg_afb)].groupby(['SAMPLE_REFID','MBGRPID','MBTESTCD','MBTSTDTL'],dropna=True).apply(lambda x:x['RESULT'])).reset_index()

b=a[a['MBGRPID'].str.contains('MGIT')]
b_=a_[a_['MBGRPID'].str.contains('LJ')]

## Function for checking samples where the MGIT and validating ZN-smear are discordant
def get_false_pos_ttp_samples(x):
    afb_res=x.loc[x['MBTESTCD']=='AFB','RESULT'].values[0]
    try:
        ttp_res=x.loc[x['MBTESTCD']=='MTB','RESULT'].values[0]
        if afb_res=='negative' and ttp_res=='positive':
            return 'false positive'

    ## If IndexError: the MGIT was negative, but a ZN smear was performed nonetheless
    except IndexError:
        pass     


false_pos_mgit_sample_idx=b.groupby(['SAMPLE_REFID','MBGRPID'],dropna=False).apply(get_false_pos_ttp_samples).reset_index()['SAMPLE_REFID'].tolist()
false_pos_lj_sample_idx=b_.groupby(['SAMPLE_REFID','MBGRPID'],dropna=False).apply(get_false_pos_ttp_samples).reset_index()['SAMPLE_REFID'].tolist()
#false_pos_mgit_idx=mb_subset_clean.loc[(mb_subset_clean['SAMPLE_REFID'].isin(false_pos_mgit_sample_idx))&\
#                                        (mb_subset['MBGRPID'].str.contains('MGIT'))].index.tolist()   



##======== SET FALSE POSITIVE MGIT & LJ RESULTS TO NEGATIVE ===============
# MGIT
filt=(mb_subset_clean['SAMPLE_REFID'].isin(false_pos_mgit_sample_idx))&\
                    (mb_subset_clean['MBTSTDTL'].str.contains('Time')).values

mb_subset_clean.loc[filt,['STD_RESULT','STD_NUM_RESULT','STD_CAT_RESULT','STD_NUM_UNITS']]=np.array([['negative',np.nan,0,np.nan]]*filt.sum())

# LJ 
filt=(mb_subset_clean['SAMPLE_REFID'].isin(false_pos_lj_sample_idx))&\
                    (mb_subset_clean['MBLNKID'].str.contains('LJ')).values

mb_subset_clean.loc[filt,['STD_RESULT','STD_NUM_RESULT','STD_CAT_RESULT','STD_NUM_UNITS']]=np.array([['negative',np.nan,0,np.nan]]*filt.sum())      
                           
                    

##======== DROP VALIDATION ZN-SMEARS OF CULTURE ISOLATES ===============
## These ZN-smears serve to check if the culture is false positive. 
#  (true for MGIT, for LJ not sure, as it is not specified in protocol)'
#  Because the information of these samples' positivity is already contained in the LJ or MGIT test result, we can drop these results.                                  

mb_subset_clean=mb_subset_clean.loc[~((mb_subset_clean['MBTESTCD']=='AFB')&\
                                    (mb_subset_clean['MBGRPID'].str.contains('MGIT|LJ'))),:]
break

##======== DETERMINE CULUTRE STATUS OF PATIENT BASED ON THE DEFINITION IN REMOX PROTOCAL MANUAL ===============
## BASED ON REMOX PROTOCOL - ANALYSIS PLAN, POINT 8 (PAGE 148/253)
## Determine culture status (positive or negative) based on the trial description rules.
def determine_culture_status(df):
    """
    Args:
        df (pd.DataFrame): Input dataframe with columns 'MBDY', 'MBSEQ', 'SAMPLE_REFID', and 'STD_RESULT'.

    Returns:
        pd.DataFrame: DataFrame with an additional 'CULTURE_STATUS' column.
    """
    # Sort the dataframe by study day ('MBDY') and sequence ('MBSEQ')
    df = df.sort_values(by=['MBDY_estimated', 'MBSEQ']).reset_index(drop=True)

    # Initialize status list, assuming first status is positive
    culture_status = ['positive']

    # Variables to track culture history
    negative_streak = 0
    last_positive_day = None

    if 'STD_RESULT_SAMPLE' in df.columns:
        std_result_colname='STD_RESULT_SAMPLE'
    else:
        std_result_colname='STD_RESULT'

    # Iterate over rows starting from the second row
    for i in range(1, len(df)):
        current_result = df.loc[i, 'STD_RESULT']
        current_day = df.loc[i, 'MBDY_estimated']

        if current_result == 'negative':
            negative_streak += 1

            # Check if three consecutive negatives achieved
            if negative_streak >= 3:
                culture_status.append('negative')
            else:
                culture_status.append(culture_status[-1])  # Maintain previous status

        elif current_result == 'positive':
            if culture_status[-1] == 'negative':
                # Determine if this positive is an isolated positive
                if (
                    last_positive_day is not None and
                    (current_day - last_positive_day) <= 91  # 13 weeks = 91 days
                ) or (
                    negative_streak < 2  # Not preceded by two negatives
                ):
                    culture_status.append('positive')
                else:
                    culture_status.append('negative')
            else:
                culture_status.append('positive')

            # Reset negative streak and update last positive day
            negative_streak = 0
            last_positive_day = current_day

    # Add the culture status column to the dataframe
    df['CULTURE_STATUS'] = culture_status
    return df


## Function to get mode of multiple test results performed on a day
def get_mode_of_result(x):
    if len(x['STD_RESULT'].mode())==1:
        return x['STD_RESULT'].mode()[0]
        
    if len(x['STD_RESULT'].mode())==2:
        return 'positive'

    if len(x['STD_RESULT'].mode())>2:
        raise ValueError('Invalid mb result! (not "negative" or "positive")')  


###---- 1. LJ-CULTURE.  ---###
#### Add culture status of patient based on the LJ-culture results
a=(mb_subset_clean.groupby(['USUBJID','MBDY_estimated', 'STUDYID','MBTESTCD','MEDIATYP','MBTSTDTL'],dropna=False,group_keys=True).apply(lambda x: x[['MBMETHOD','MBSEQ','STD_NUM_RESULT','STD_CAT_RESULT','STD_NUM_UNITS','STD_RESULT','SAMPLE_REFID']]))
b=a.reset_index()


print('Extracting LJ samples')

lj_cultures=b[~(b['MEDIATYP'].isna()) & (b['MBTESTCD'].str.contains('MTB'))]

## Clean up STD_CAT RESULT and STD_NUM_RESULT columns 
lj_cultures['STD_NUM_RESULT']=lj_cultures['STD_NUM_RESULT'].replace('nan',np.nan)
lj_cultures['STD_CAT_RESULT']=lj_cultures['STD_CAT_RESULT'].replace('0','NEGATIVE')
lj_cultures.loc[~lj_cultures['STD_NUM_RESULT'].isna(),'STD_CAT_RESULT']=np.nan


lj_cultures['STD_CAT_ORDINAL_RESULT']=np.nan

tb21_LJ_ordinal_conversion_table={'NEGATIVE':0,
                                        '1+':1,
                                        '2+':2,
                                        '3+':3}
                                        

lj_cultures.loc[lj_cultures['MBTSTDTL']=='Colony Count, Categorical',
                'STD_CAT_ORDINAL_RESULT']=lj_cultures.loc[lj_cultures['MBTSTDTL']=='Colony Count, Categorical','STD_CAT_RESULT'].map(tb21_LJ_ordinal_conversion_table)


## Get mode of binary LJ-culture results performed on the same day 
#  If mode positive ==> extract the highest categorical or numerical results
#. If mode negative ==> set both categorical and numerical result to 0

def get_mode_of_lj_result(x):
    if len(x['STD_RESULT'].mode())==1:
        
        res = x['STD_RESULT'].mode()[0]        
        if res=='positive':
            res_cat=x['STD_CAT_ORDINAL_RESULT'].max()
            res_num=x['STD_NUM_RESULT'].max()
        if res=='negative':
            res_cat=0
            res_num=0        
        return pd.Series([res,res_cat,res_num])
        
    if len(x['STD_RESULT'].mode())==2:
        res='positive'
        res_cat=x['STD_CAT_ORDINAL_RESULT'].max()
        res_num=x['STD_NUM_RESULT'].max()
        return pd.Series([res,res_cat,res_num])

    if len(x['RESULT'].mode())>2:
        raise ValueError('Invalid mb result! (not "negative" or "positive")')  

## The result of a sample is reported twice in many cases: once as Cultre growth (positive,negative) and once as a Bacterial Count (categorical (1+/2+/3+... or CFU)
## These two values should always be equal to each other 
## ==> However, get mode of the growth & count result from the same sample 
## ==> if they are discordant (one is neg. the other is pos.), just assume that the sample was positive 
#lj_sample_results=lj_cultures.groupby('SAMPLE_REFID').apply(get_mode_of_result)
#lj_cultures['STD_RESULT_SAMPLE']=lj_sample_results.loc[lj_cultures['SAMPLE_REFID'].values.tolist()].values


## Culture growth: 1927 patients
## Colony Count + Colony Count Categorical: 1871 patients

## Based on the Remox cluture status criteria, add a column with culture status based on the LJ-cultures
## Only use the Culture Growth results (they a
#lj_cultures = lj_cultures[(lj_cultures['MBTSTDTL']=='Culture Growth')].groupby('USUBJID').apply(lambda x: determine_culture_status(x))
#print('culture status determined')

## Based on the Remox culture status criteria, add a column with culture status based on the LJ-cultures
## Only use the Culture Growth results (they a
lj_cultures_cult_status = lj_cultures[(lj_cultures['MBTSTDTL']=='Culture Growth')].groupby('USUBJID').apply(lambda x: determine_culture_status(x))

## If multiple samples were done on a day, extract the CULTURE_STATUS of that sample, which has the highest MBSEQ (== was done latest)
## ==> CULTURE_STATUS was determined using the MBSEQ order
lj_cult_status_per_day=lj_cultures_cult_status.drop(columns=['USUBJID']).reset_index()[['USUBJID','MBDY_estimated','SAMPLE_REFID',\
                                            'MBSEQ','CULTURE_STATUS']].groupby(['USUBJID','MBDY_estimated']).apply(lambda x:x.loc[x['MBSEQ']==x['MBSEQ'].max(),'CULTURE_STATUS'])
print('lj_cult_status_per_day done')

## If multiple samples were done on a day, extract the LJ result for that day, by calculating the mode of the multiple samples
lj_results_per_day = lj_cultures.reset_index().groupby(['USUBJID','STUDYID','MBDY_estimated'],dropna=False).apply(get_mode_of_lj_result)
print('lj_results_per_day done')


### ADD CULTURE STATUS TO THE LJ PER-DAY RESULT DATAFRAME CREATING A DATAFRAME CONTAINING:
## 1. MODE OF ALL LJ-SAMPLES ON GIVEN VISIT DAY AS AN AGGREGATED RESULT OF THE SAMPLE FROM THE VISIT
## 2. CULTURE STATUS FOR GIVEN DAY, DERIVED FROM ALL PREVIOUS LJ-TESTS, AND AGGREGATED INTO ONE VALUE PER STUDY DAY
lj_results_per_day=lj_results_per_day.reset_index()
lj_results_per_day = lj_results_per_day.rename(columns={0:'STD_RESULT',1:'STD_CAT_ORDINAL_RESULT',2:'STD_NUM_RESULT'})
lj_results_per_day['CULTURE_STATUS']=lj_cult_status_per_day.values
lj_results_per_day['MBMETHOD']='MICROBIAL CULTURE, SOLID' 
lj_results_per_day['STD_TEST']='LJ-culture'
lj_results_per_day['STD_NUM_UNIT']=np.nan
lj_results_per_day.loc[~lj_results_per_day['STD_NUM_RESULT'].isna(),'STD_NUM_UNIT']='CFU'




###--- 2. MGIT.  ---###
print('Extracting MGIT samples')

#### Add culture status of patient based on the MGIT-culture results
mgit_cultures = b[(b['MBTESTCD']=='MTB')&(b['MBTSTDTL']=='Time to Detection')]

## One sample has its REFID duplicated, as sample from DAY 137 and another from day 159 have the same REFID ==> probably a typographical error
## Change the refid for one the samples
mgit_cultures.loc[21631,'SAMPLE_REFID']='TB-1021_201049_1'


## Based on the Remox cluture status criteria, add a column with culture status based on the MGIT-cultures
mgit_cultures = mgit_cultures.groupby('USUBJID').apply(lambda x: determine_culture_status(x))

## If multiple samples were done on a day, extract the CULTURE_STATUS of that sample, which has the highest MBSEQ (== was done latest)
## ==> CULTURE_STATUS was determined using the MBSEQ order
mgit_cult_status_per_day=mgit_cultures.drop(columns=['USUBJID']).reset_index()[['USUBJID','MBDY_estimated','SAMPLE_REFID',\
                                        'MBSEQ','CULTURE_STATUS']].groupby(['USUBJID','MBDY_estimated']).apply(lambda x:x.loc[x['MBSEQ']==x['MBSEQ'].max(),'CULTURE_STATUS'])

## Function to get mode of multiple test results performed on a day
def get_mode_of_result(x):
    if len(x['STD_RESULT'].mode())==1:
        return x['STD_RESULT'].mode()[0]
        
    if len(x['STD_RESULT'].mode())==2:
        return 'positive'

    if len(x['STD_RESULT'].mode())>2:
        raise ValueError('Invalid mb result! (not "negative" or "positive")')  

## If multiple samples were done on a day, extract the MGIT result for that day, by calculating the mode of the multiple samples
mgit_results_per_day = mgit_cultures.drop(columns=['USUBJID']).reset_index().groupby(['USUBJID','STUDYID','MBDY_estimated'],dropna=False).apply(get_mode_of_result)


### GET MEAN OF THE MGIT TTP VALUES ON A GIVEN DAY
#. ==> if there are more negative than positive MGITs on one day (across all samples taken on a given day!) ==> take MGIT as negative
#  ==> if there are more positive than negative results on a day ==> taken average of positive results (assume the negative as false negative)
#  ==> if there are equal positive than negative results on a day ==> taken average of positive results (assume the worse scenario of bacteria being present)

## Function to get mean of multiple MGIT test results performed on a day
def get_mean_of_num_test_result(x):
    
    if len(x['STD_RESULT'].mode())==1:
        
        if x['STD_RESULT'].mode()[0]=='negative':
            return np.nan
            
        if x['STD_RESULT'].mode()[0]=='positive':
            return x.loc[~x['STD_NUM_RESULT'].isna(),'STD_NUM_RESULT'].astype(float).mean()

            
    if len(x['STD_RESULT'].mode())==2:
        return x.loc[~x['STD_NUM_RESULT'].isna(),'STD_NUM_RESULT'].astype(float).mean()

    if len(x['STD_RESULT'].mode())>2:
        raise ValueError('Invalid mb result! (not "negative" or "positive")')  

mgit_ttp_per_day=mgit_cultures.drop(columns=['USUBJID']).reset_index().groupby(['USUBJID','STUDYID','MBDY_estimated'],dropna=False).apply(get_mean_of_num_test_result)
mgit_ttp_per_day=mgit_ttp_per_day.reset_index().rename(columns={0:'STD_NUM_RESULT'})
mgit_ttp_per_day['STD_NUM_UNIT']=np.nan
mgit_ttp_per_day.loc[~mgit_ttp_per_day['STD_NUM_RESULT'].isna(),'STD_NUM_UNIT']='DAYS'


### ADD CULTURE STATUS TO THE LJ PER-DAY RESULT DATAFRAME CREATING A DATAFRAME CONTAINING:
## 1. MODE OF ALL LJ-SAMPLES ON GIVEN VISIT DAY AS AN AGGREGATED RESULT OF THE SAMPLE FROM THE VISIT
## 2. CULTURE STATUS FOR GIVEN DAY, DERIVED FROM ALL PREVIOUS LJ-TESTS, AND AGGREGATED INTO ONE VALUE PER STUDY DAY
mgit_results_per_day=pd.concat([mgit_results_per_day,mgit_ttp_per_day.set_index(['USUBJID','STUDYID','MBDY_estimated'])],axis=1).reset_index()
mgit_results_per_day = mgit_results_per_day.rename(columns={0:'STD_RESULT'})
mgit_results_per_day['CULTURE_STATUS']=mgit_cult_status_per_day.values
mgit_results_per_day['MBMETHOD']='MICROBIAL CULTURE, LIQUID'
mgit_results_per_day['STD_TEST']='MGIT'


###--- 3. AFB  ---###
print('Extracting AFB samples')
## If multiple samples were done on a day, extract the AFB result for that day, by calculating the mode of the multiple samples
afb_results=b.loc[b['MBTESTCD']=='AFB',:]

## Create new column with converting ZN-smear results to ordinal levels
tb_22_21_ZN_ordinal_conversion_table={'NEGATIVE':0,
                                        '1+':1,
                                        '2+':2,
                                        '3+':3,
                                        '4+':4}

afb_results['STD_CAT_ORDINAL_RESULT']=afb_results['STD_CAT_RESULT'].map(tb_22_21_ZN_ordinal_conversion_table)

## Get mode of binary results
def get_mode_of_ZN_result(x):
    if len(x['STD_RESULT'].mode())==1:
        res = x['STD_RESULT'].mode()[0]
        
        if res=='positive':
            res_cat=x['STD_CAT_ORDINAL_RESULT'].max()
        if res=='negative':
            res_cat=0#x['STD_CAT_ORDINAL_RESULT'].min()
        return pd.Series([res,res_cat])
        
    if len(x['STD_RESULT'].mode())==2:
        return pd.Series(['positive',x['STD_CAT_ORDINAL_RESULT'].max()])

    if len(x['STD_RESULT'].mode())>2:
        raise ValueError('Invalid mb result! (not "negative" or "positive")')  


afb_results_per_day=afb_results.groupby(['USUBJID','STUDYID','MBDY_estimated'],dropna=False).apply(get_mode_of_ZN_result)
afb_results_per_day=afb_results_per_day.reset_index()
afb_results_per_day = afb_results_per_day.rename(columns={0:'STD_RESULT',1:'STD_CAT_ORDINAL_RESULT'})
afb_results_per_day['STD_CAT_RESULT'] = afb_results_per_day['STD_CAT_ORDINAL_RESULT'].map(dict(zip(tb_22_21_ZN_ordinal_conversion_table.values(),tb_22_21_ZN_ordinal_conversion_table.keys())))
afb_results_per_day['STD_TEST']='ZN-smear'


###--- 4. AccuProbe  ---###
print('Extracting AccuProbe samples')
## If multiple samples were done on a day, extract the AccuProbe result for that day, by calculating the mode of the multiple samples
accu_results=b.loc[b['MBTESTCD']=='MTBCMPLX',:]#.groupby(['USUBJID','STUDYID','MBDY_estimated'],dropna=False).apply(get_mode_of_result

def get_mode_of_result(x):
    if len(x['STD_RESULT'].mode())==1:
        return x['STD_RESULT'].mode()[0]
        
    if len(x['STD_RESULT'].mode())==2:
        return 'positive'

    if len(x['STD_RESULT'].mode())>2:
        raise ValueError('Invalid mb result! (not "negative" or "positive")')
        
accu_results_per_day=accu_results.groupby(['USUBJID','STUDYID','MBDY_estimated'],dropna=False).apply(get_mode_of_result)
accu_results_per_day=accu_results_per_day.reset_index()
accu_results_per_day = accu_results_per_day.rename(columns={0:'STD_RESULT'})
accu_results_per_day['STD_TEST']='AccuProbe'


###====== CONCATENATE DATA FROM DIFFERENT TEST METHODS INTO ONE DATAFRAME ===== ###
tb_21_std_res = pd.concat([afb_results_per_day,lj_results_per_day,mgit_results_per_day,accu_results_per_day],axis=0)




Num of initial samples 176592
176592
Num of samples after dropping contaminated MGIT + blood agar samples 154066


/tmp/ipykernel_211693/2231927218.py:54: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  a=(mb_subset_clean[mb_subset_clean['SAMPLE_REFID'].isin(mgit_samples_with_neg_afb)].groupby(['SAMPLE_REFID','MBGRPID','MBTESTCD','MBTSTDTL'],dropna=True).apply(lambda x:x['RESULT'])).reset_index()
/tmp/ipykernel_211693/2231927218.py:55: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  a_=(mb_subset_clean[mb_subset_clean['SAMPLE_REFID']

SyntaxError: 'break' outside loop (2231927218.py, line 102)

In [8]:
mb_subset_clean.columns

Index(['STUDYID', 'DOMAIN', 'USUBJID', 'SPDEVID', 'MBSEQ', 'MBGRPID',
       'MBREFID', 'MBSPID', 'MBLNKID', 'MBTESTCD', 'MBTEST', 'MBTSTDTL',
       'MBORRES', 'MBORRESU', 'MBSTRESC', 'MBSTRESN', 'MBSTRESU', 'MBSPEC',
       'MBMETHOD', 'MBDRVFL', 'VISITNUM', 'VISIT', 'EPOCH', 'MBDTC', 'MBDY',
       'MBEVINTX', 'MEDIATYP', 'COMMENT', 'RESULT', 'STD_NUM_RESULT',
       'STD_NUM_UNITS', 'STD_CAT_RESULT', 'STD_CAT_UNITS', 'MBDY_estimated',
       'ARM', 'SAMPLE_REFID', 'STD_RESULT'],
      dtype='object')

In [11]:
mb_subset_clean['MBTSTDTL'].value_counts(dropna=False)

MBTSTDTL
Categorical Count            37652
Culture Growth               37358
Time to Detection            34734
Colony Count, Categorical     8495
Identification                3733
Colony Count                  2707
Name: count, dtype: int64

In [19]:
mb_subset_clean[mb_subset_clean['MBSTRESN'].isna()]#['MBTSTDTL'].value_counts()

,STUDYID,DOMAIN,USUBJID,SPDEVID,MBSEQ,MBGRPID,MBREFID,MBSPID,MBLNKID,MBTESTCD,...,COMMENT,RESULT,STD_NUM_RESULT,STD_NUM_UNITS,STD_CAT_RESULT,STD_CAT_UNITS,MBDY_estimated,ARM,SAMPLE_REFID,STD_RESULT
2,TB-1021,MB,TB-1021/1098684,NaN,3,NaN,200156,NaN,NaN,MTB,...,NaN,negative,NaN,NaN,NEGATIVE,NaN,1.0,2EMRZ/2MR,TB-1021_200156,negative
3,TB-1021,MB,TB-1021/1098908,NaN,3,NaN,200157,NaN,NaN,MTB,...,NaN,negative,NaN,NaN,NEGATIVE,NaN,1.0,2EMRZ/2MR,TB-1021_200157,negative
4,TB-1021,MB,TB-1021/1093230,NaN,4,LJ0000,200335,NaN,LJ0000,MTB,...,NaN,positive,NaN,NaN,2+,NaN,-5.0,NaN,TB-1021_200335,positive
5,TB-1021,MB,TB-1021/1098684,MGIT,4,NaN,200156,430140961918,NaN,MTB,...,NaN,negative,NaN,DAYS,>42,DAYS,1.0,NaN,TB-1021_200156,negative
6,TB-1021,MB,TB-1021/1098908,MGIT,4,NaN,200157,430140961908,NaN,MTB,...,NaN,negative,NaN,DAYS,>42,DAYS,1.0,NaN,TB-1021_200157,negative
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390885,TB-1021,MB,TB-1021/1089120,NaN,24,NaN,200029,NaN,NaN,MTB,...,NaN,negative,NaN,NaN,NEGATIVE,NaN,15.0,NaN,TB-1021_200029,negative
390886,TB-1021,MB,TB-1021/1013413,NaN,27,LJ0300,400072,NaN,LJ0300,MTB,...,NaN,positive,NaN,NaN,POSITIVE,NaN,14.0,NaN,TB-1021_400072,positive
390888,TB-1021,MB,TB-1021/1101411,NaN,30,LJ0400,200193,NaN,LJ0400,MTB,...,NaN,positive,NaN,NaN,2+,NaN,21.0,NaN,TB-1021_200193,positive
390889,TB-1021,MB,TB-1021/1101411,NaN,31,LJ0400,200193,NaN,LJ0400,MTB,...,NaN,positive,NaN,NaN,POSITIVE,NaN,21.0,NaN,TB-1021_200193,positive


In [27]:
tb_21_std_res[~tb_21_std_res['STD_NUM_RESULT'].isna()]['STD_TEST'].value_counts()

STD_TEST
LJ-culture    20469
MGIT          14655
Name: count, dtype: int64

In [ ]:
mb_subset['MBTSTDTL'].unique()

In [ ]:
accu_results

In [ ]:
mb_subset.loc[(mb_subset['USUBJID']=='TB-1021/1003959')\
            #&(mb_subset['MBGRPID'].str.contains('MGIT'))\
            &(mb_subset['MBTESTCD']=='MTB'),].sort_values('MBDY')

In [6]:
tb_21_std_res#[tb_21_std_res['USUBJID']=='TB-1021/1006439']

,USUBJID,STUDYID,MBDY_estimated,STD_RESULT,STD_CAT_ORDINAL_RESULT,STD_CAT_RESULT,STD_TEST,STD_NUM_RESULT,CULTURE_STATUS,MBMETHOD,STD_NUM_UNIT
0,TB-1021/1001485,TB-1021,-2.0,positive,2.0,2+,ZN-smear,NaN,NaN,NaN,NaN
1,TB-1021/1001485,TB-1021,1.0,positive,4.0,4+,ZN-smear,NaN,NaN,NaN,NaN
2,TB-1021/1001485,TB-1021,8.0,positive,1.0,1+,ZN-smear,NaN,NaN,NaN,NaN
3,TB-1021/1001485,TB-1021,15.0,positive,4.0,4+,ZN-smear,NaN,NaN,NaN,NaN
4,TB-1021/1001485,TB-1021,22.0,positive,4.0,4+,ZN-smear,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
3264,TB-1021/3027114,TB-1021,454.0,negative,NaN,NaN,AccuProbe,NaN,NaN,NaN,NaN
3265,TB-1021/3033214,TB-1021,1.0,positive,NaN,NaN,AccuProbe,NaN,NaN,NaN,NaN
3266,TB-1021/3040350,TB-1021,1.0,positive,NaN,NaN,AccuProbe,NaN,NaN,NaN,NaN
3267,TB-1021/3040350,TB-1021,153.0,negative,NaN,NaN,AccuProbe,NaN,NaN,NaN,NaN


### Save patient ID of contaminated & false positive samples for TB-1021

In [ ]:
contam_ttp_sample_ids = [x.split('TB-1021_')[-1] for x in contam_ttp_sample_ids]
false_pos_mgit_sample_idx = [x.split('TB-1021_')[-1] for x in false_pos_mgit_sample_idx]


fn='../data/tb_1021_contam_ttp_sample_ids.pickle'
with open(fn, 'wb') as handle:
    pickle.dump(contam_ttp_sample_ids, handle)


fn='../data/tb_1021_false_pos_mgit_sample_idx.pickle'
with open(fn, 'wb') as handle:
    pickle.dump(false_pos_mgit_sample_idx, handle)

# TB-1020

In [ ]:
# Finish this - ZN-smear : conversion to CDC (tb21)

In [15]:
study_name='TB-1020'
mb_subset=mb[mb['STUDYID']==study_name].dropna(how='all',axis=1)
mb_subset[['STD_RESULT']]=mb_subset[['RESULT']].values
print('Num of initial samples',len(mb_subset))



## CONVERT ZN-SMEAR RESULTS TO THE CDC QUANTIFICATION LEVELS (based on: remox-laboratory-manual.pdf page 15, 
#. also used in publication: https://www.nature.com/articles/s41591-018-0224-2#Sec9
tb_22_21_ZN_conversion_table={'NEGATIVE':'NEGATIVE',
                                'SCANTY':'1+',
                                 '1+':'2+',
                                 '2+':'3+',
                                 '3+':'4+'}

## Create new column with converting ZN-smear results to ordinal levels
tb_22_21_ZN_ordinal_conversion_table={'NEGATIVE':0,
                                        '1+':1,
                                        '2+':2,
                                        '3+':3,
                                        '4+':4}

filt=(mb_subset['MBTESTCD']=='AFB')\
                 &(mb_subset['MBMETHOD'].str.contains('ZIEHL'))

mb_subset.loc[filt,'STD_CAT_RESULT']=mb_subset.loc[filt,'STD_CAT_RESULT'].map(tb_22_21_ZN_conversion_table)
mb_subset['STD_CAT_ORDINAL_RESULT']=np.nan
mb_subset.loc[filt,'STD_CAT_ORDINAL_RESULT'] = mb_subset.loc[filt,'STD_CAT_RESULT'].map(tb_22_21_ZN_ordinal_conversion_table)






def determine_culture_status(df):
    """
    Args:
        df (pd.DataFrame): Input dataframe with columns 'MBDY', 'MBSEQ', 'SAMPLE_REFID', and 'STD_RESULT'.

    Returns:
        pd.DataFrame: DataFrame with an additional 'CULTURE_STATUS' column.
    """
    # Sort the dataframe by study day ('MBDY') and sequence ('MBSEQ')
    df = df.sort_values(by=['MBDY_estimated', 'MBSEQ']).reset_index(drop=True)

    # Initialize status list, assuming first status is positive
    #culture_status = ['positive']
    culture_status = [df['STD_RESULT'].iloc[0]]

    # Variables to track culture history
    negative_streak = 0
    last_positive_day = None

    if 'STD_RESULT_SAMPLE' in df.columns:
        std_result_colname='STD_RESULT_SAMPLE'
    else:
        std_result_colname='STD_RESULT'

    # Iterate over rows starting from the second row
    for i in range(1, len(df)):
        current_result = df.loc[i, 'STD_RESULT']
        current_day = df.loc[i, 'MBDY_estimated']

        if current_result == 'negative':
            negative_streak += 1

            # Check if three consecutive negatives achieved
            if negative_streak >= 3:
                culture_status.append('negative')
            else:
                culture_status.append(culture_status[-1])  # Maintain previous status

        elif current_result == 'positive':
            if culture_status[-1] == 'negative':
                # Determine if this positive is an isolated positive
                if (
                    last_positive_day is not None and
                    (current_day - last_positive_day) <= 91  # 13 weeks = 91 days
                ) or (
                    negative_streak < 2  # Not preceded by two negatives
                ):
                    culture_status.append('positive')
                else:
                    culture_status.append('negative')
            else:
                culture_status.append('positive')

            # Reset negative streak and update last positive day
            negative_streak = 0
            last_positive_day = current_day

    # Add the culture status column to the dataframe
    df['CULTURE_STATUS'] = culture_status
    return df


## Based on the Remox cluture status criteria, add a column with culture status based on the MGIT-cultures
#mgit_cultures = mgit_cultures.groupby('USUBJID').apply(lambda x: determine_culture_status(x))
mgit_cultures = mb_subset.loc[mb_subset['SPDEVID'].str.contains('MGIT',na=False),:]
mgit_cultures = mgit_cultures.groupby('USUBJID').apply(lambda x: determine_culture_status(x))

## If multiple samples were done on a day, extract the CULTURE_STATUS of that sample, which has the highest MBSEQ (== was done latest)
## ==> CULTURE_STATUS was determined using the MBSEQ order
mgit_cult_status_per_day=mgit_cultures.drop(columns=['USUBJID']).reset_index()[['USUBJID','MBDY_estimated','SAMPLE_REFID',\
                                        'MBSEQ','CULTURE_STATUS']].groupby(['USUBJID','MBDY_estimated']).apply(lambda x:x.loc[x['MBSEQ']==x['MBSEQ'].max(),'CULTURE_STATUS'])






## Function to get mean of multiple MGIT test results performed on a day
def get_mean_of_num_test_result(x):
    
    if len(x['RESULT'].mode())==1:
        
        if x['RESULT'].mode()[0]=='negative':
            std_num_res, std_res = np.nan, 'negative'
            return pd.Series(data=[std_num_res, std_res])
            
        if x['RESULT'].mode()[0]=='positive':
            std_num_res, std_res = x.loc[~x['STD_NUM_RESULT'].isna(),'STD_NUM_RESULT'].astype(float).mean(), 'positive'
            return pd.Series(data=[std_num_res, std_res])

    ## If mode is equal negative and positive ==> take positive
    if len(x['RESULT'].mode())==2:
        std_num_res, std_res = x.loc[~x['STD_NUM_RESULT'].isna(),'STD_NUM_RESULT'].astype(float).mean(), 'positive'
        return pd.Series(data=[std_num_res, std_res]) 

    if len(x['RESULT'].mode())>2:
        raise ValueError('Invalid mb result! (not "negative" or "positive")')  

## MGIT results are reported as categorical values, but for the baseline and first follow-up visit, 
#  the TTP values are also recorded 
#   ==> Based on mode of results on a given day:
#        -If mode is negative ==> take nagtive as value
#.        If mode is positive or ambiguous ==> take positive + mean of MGIT values


mgit_res = mb_subset.loc[mb_subset['SPDEVID'].str.contains('MGIT',na=False),:].groupby(['MBREFID','USUBJID','MBDY_estimated']).apply(lambda x:x[['STD_NUM_RESULT','RESULT']])
mgit_results_per_day=mgit_res.groupby(['USUBJID','MBDY_estimated']).apply(lambda x:get_mean_of_num_test_result(x))
mgit_results_per_day.columns=['STD_NUM_RESULT','STD_RESULT']
mgit_results_per_day['STD_TEST']='MGIT'
mgit_results_per_day['CULTURE_STATUS']=mgit_cult_status_per_day.values
#mgit_results_per_day.loc[~mgit_results_per_day['STD_NUM_RESULT'].isna(),'STD_NUM_UNIT']='DAYS'
mgit_results_per_day.loc[~mgit_results_per_day['STD_NUM_RESULT'].isna(),'STD_NUM_UNIT']='DAYS'






## If multiple samples were done on a day, extract the AFB result for that day, by calculating the mode of the multiple samples
afb_results=mb_subset[mb_subset['MBTESTCD']=='AFB']

## Create new column with converting ZN-smear results to ordinal levels
tb_22_21_ZN_ordinal_conversion_table={'NEGATIVE':0,
                                        '1+':1,
                                        '2+':2,
                                        '3+':3,
                                        '4+':4}

afb_results['STD_CAT_ORDINAL_RESULT']=afb_results['STD_CAT_RESULT'].map(tb_22_21_ZN_ordinal_conversion_table)

## Get mode of binary results
def get_mode_of_ZN_result(x):
    if len(x['STD_RESULT'].mode())==1:
        res = x['STD_RESULT'].mode()[0]
        
        if res=='positive':
            res_cat=x['STD_CAT_ORDINAL_RESULT'].max()
        if res=='negative':
            res_cat=0#x['STD_CAT_ORDINAL_RESULT'].min()
        return pd.Series([res,res_cat])
        
    if len(x['STD_RESULT'].mode())==2:
        return pd.Series(['positive',x['STD_CAT_ORDINAL_RESULT'].max()])

    if len(x['STD_RESULT'].mode())>2:
        raise ValueError('Invalid mb result! (not "negative" or "positive")')  


afb_results_per_day=afb_results.groupby(['USUBJID','STUDYID','MBDY_estimated'],dropna=False).apply(get_mode_of_ZN_result)
afb_results_per_day=afb_results_per_day.reset_index()
afb_results_per_day = afb_results_per_day.rename(columns={0:'STD_RESULT',1:'STD_CAT_ORDINAL_RESULT'})
afb_results_per_day['STD_CAT_RESULT'] = afb_results_per_day['STD_CAT_ORDINAL_RESULT'].map(dict(zip(tb_22_21_ZN_ordinal_conversion_table.values(),tb_22_21_ZN_ordinal_conversion_table.keys())))
afb_results_per_day['STD_TEST']='ZN-smear'
afb_results_per_day = afb_results_per_day.set_index(['USUBJID','MBDY_estimated'])




def get_mode_of_result(x):
    if len(x['RESULT'].mode())==1:
        return x['RESULT'].mode()[0]
        
    if len(x['RESULT'].mode())==2:
        return 'positive'

    if len(x['RESULT'].mode())>2:
        raise ValueError('Invalid mb result! (not "negative" or "positive")') 



id_results=mb_subset[mb_subset['MBTESTCD']=='MTBCMPLX']

id_results_per_day = id_results.groupby(['USUBJID','MBDY_estimated'],dropna=False).apply(get_mode_of_result).to_frame()
id_results_per_day.columns=['STD_RESULT']
id_results_per_day['STD_TEST'] = 'MTB-complex'

###====== CONCATENATE DATA FROM DIFFERENT TEST METHODS INTO ONE DATAFRAME ===== ###
tb_20_std_res = pd.concat([afb_results_per_day,mgit_results_per_day,id_results_per_day],axis=0)

tb_20_std_res['STUDYID']='TB-1020'
tb_20_std_res = tb_20_std_res.reset_index()

 

Num of initial samples 25169


/tmp/ipykernel_184321/2585867222.py:100: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mgit_cultures = mgit_cultures.groupby('USUBJID').apply(lambda x: determine_culture_status(x))
/tmp/ipykernel_184321/2585867222.py:104: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mgit_cult_status_per_day=mgit_cultures.drop(columns=['USUBJID']).reset_index()[['USUBJID','MBDY_estimated','SAMPLE_REFID',\
/tmp/ipykernel_184321/258586

In [18]:
#tb_20_std_res[~tb_20_std_res['STD_NUM_UNIT'].isna()]

,USUBJID,MBDY_estimated,STUDYID,STD_RESULT,STD_CAT_ORDINAL_RESULT,STD_CAT_RESULT,STD_TEST,STD_NUM_RESULT,CULTURE_STATUS,STD_NUM_UNIT
9904,TB-1020/1002,1.0,TB-1020,positive,NaN,NaN,MGIT,8.0,positive,DAYS
9910,TB-1020/1002,239.0,TB-1020,positive,NaN,NaN,MGIT,38.0,positive,DAYS
9916,TB-1020/1003,1.0,TB-1020,positive,NaN,NaN,MGIT,18.0,positive,DAYS
9922,TB-1020/1003,212.0,TB-1020,positive,NaN,NaN,MGIT,24.0,positive,DAYS
9928,TB-1020/1004,4.0,TB-1020,positive,NaN,NaN,MGIT,13.0,positive,DAYS
...,...,...,...,...,...,...,...,...,...,...
15525,TB-1020/4252,1.0,TB-1020,positive,NaN,NaN,MGIT,10.0,positive,DAYS
15538,TB-1020/4253,1.0,TB-1020,positive,NaN,NaN,MGIT,8.0,positive,DAYS
15542,TB-1020/4253,151.0,TB-1020,positive,NaN,NaN,MGIT,23.0,positive,DAYS
15554,TB-1020/4254,1.0,TB-1020,positive,NaN,NaN,MGIT,28.5,positive,DAYS


# TB-1018

In [19]:
study_name='TB-1018'
mb_subset=mb[mb['STUDYID']==study_name].dropna(how='all',axis=1)
mb_subset[['STD_RESULT']]=mb_subset[['RESULT']].values
print('Num of initial samples',len(mb_subset))

mb_subset.groupby(['STUDYID','MBTESTCD','MEDIATYP'],dropna=False).apply(lambda x: (x['MBTSTDTL'].value_counts()))#.reset_index()


##======== IDENTIFY CONTAMINATED POSITIVE MGIT SAMPLES ===============
#  Based on: https://www.nejm.org/doi/suppl/10.1056/NEJMoa1901814/suppl_file/nejmoa1901814_protocol.pdf page 410-455
#  MGIT :450-455
#  AFB: 433-436

## IMPORTANT: 
#  -POSITIVE MGIT RESULTS ARE REPORTED AS: TTP + CULTURE GROWTH
#  -NEGATIVE MGIT RESULTS ARE REPORTED AS: ONLY CULTURE GROWTH

## 1. Positive MGIT results are getting tested for contamination by inoculating blood agar with a small sample 
#     from the MGIT tube 
#  2. If the blood agar ('MBTESTCD'=='NONMTB') is not negative ('MBSTRESC'!='NEGATIVE (MGIT RESULT VALID)'), 
#     the MGIT is considered contaminated
cont_pos_ttp_sample_ids=mb_subset.loc[(mb_subset['MBTESTCD']=='NONMTB')\
                                         &(mb_subset['MBSTRESC']!='NEGATIVE (MGIT RESULT VALID)')\
                                        ,'SAMPLE_REFID'].dropna().unique().tolist()

## 3. Check for samples, where Blood Agar culturing was not done ==> based on protocol, these should be considered as invalid 
#     ==> we have to infer them by subtracting the not-contaminated MGIT sample IDs from all MGIT sample IDs
all_pos_ttp_sample_ids=mb_subset.loc[(mb_subset['SPDEVID'].str.contains('MGIT',na=False))\
                                    &(mb_subset['MBMETHOD'].str.contains('MICROBIAL CULTURE, LIQUID',na=False))\
                                    &(mb_subset['MBSTRESC']!='NEGATIVE')\
                                     ,'SAMPLE_REFID'].unique() 

ttp_with_blood_agar_sample_ids=mb_subset.loc[(mb_subset['MBTESTCD']=='NONMTB')\
                                            #&(mb_subset['MBSTRESC']=='NEGATIVE (MGIT RESULT VALID)')\
                                            ,'SAMPLE_REFID'].dropna().unique()                                       

## Get samplesIds where no blood agar is available
pos_ttp_wo_blood_agar_sample_ids=list(set(all_pos_ttp_sample_ids) - set(ttp_with_blood_agar_sample_ids))

## Invalid MGIT sample ids = contaminated samples + samples without blood agar measurement
invalid_ttp_sample_ids = cont_pos_ttp_sample_ids + pos_ttp_wo_blood_agar_sample_ids

print(len(mb_subset))



##======= DROP CONTAMINATED POSITIVE MGIT SAMPLES + 
#         LINKED ZN-SMEAR RESULTS PERFORMED ON CONT.MGIT + 
#         BLOOD AGAR MEASUREMENTS OF VALIDATION ===============
cont_ttp_idx=mb_subset.loc[(mb_subset['SAMPLE_REFID'].isin(cont_pos_ttp_sample_ids))
                           &(mb_subset['SPDEVID'].str.contains('MGIT'))].index.tolist()

inv_ttp_idx=mb_subset.loc[(mb_subset['SAMPLE_REFID'].isin(invalid_ttp_sample_ids))\
                            &(mb_subset['SPDEVID'].str.contains('MGIT',na=False))\
                            &(mb_subset['MBMETHOD'].str.contains('MICROBIAL CULTURE, LIQUID',na=False))].index.tolist()

inv_afb_idx=mb_subset.loc[(mb_subset['SAMPLE_REFID'].isin(invalid_ttp_sample_ids))\
                            &(mb_subset['MBTESTCD'].str.contains('AFB',na=False))].index.tolist()

blood_agar_idx=mb_subset.loc[(mb['MBTESTCD']=='NONMTB')].index.tolist()   
mb_subset_clean=mb_subset.drop(index=inv_ttp_idx + blood_agar_idx +inv_afb_idx)
print('Num of samples after dropping contaminated MGIT + blood agar samples',len(mb_subset_clean))


##======== IDENTIFY CONTAMINATED/ MGIT SAMPLES, BASED ON COMMENTS OF THE MICROBIOLOGY LAB ===============
## List containing positive samples' comments, which describe some issues with the sample, but don't mean they are contaminated
not_cont_comments=['ZN REPEATED AND STILL NO AFB SEEN, BUT THE ID TEST CONFIRMED PRESENCE OF MTB COMPLEX.',
                    'MGIT TUBE NOT REGISTERED ON EPICENTRE IN ERROR, THEREFORE NO RESULT TIME',
                    'RESULT TIME NOT AVAILABLE DUE TO PROBLEM WITH EPICENTRE DATABASE',
                    'RESULT TIME NOT AVAILABLE DUE TO EPICENTER',
                    'RESULT TIME NOT AVAILABLE DUE TO EPICENTER ERROR.',
                    'RESULT TIME NOT AVAILABLE DUE TO EPICENTER ERROR',
                    'ZN WAS REPEATED AND STILL NO AFB SEEN, BUT ID TEST CONFIRMED PRESENCE OF MTB COMPLEX. ZN WAS REPEATED AGAIN AFTER REPORTING AND ZN AFB PRESENT',
                    'FALSE POSITIVE']

## Take samples, that are positive, their comment is not in the list & is not NaN (which means the results is valid)
cont_sample_sample_ids=mb_subset_clean.loc[(~mb_subset_clean['COMMENT'].isin(not_cont_comments))\
                                    &(~mb_subset_clean['COMMENT'].isna())\
                                    &(mb_subset_clean['RESULT']=='positive')\
                                    ,'SAMPLE_REFID'].unique().tolist()


mb_subset_clean=mb_subset_clean[~mb_subset_clean['SAMPLE_REFID'].isin(cont_sample_sample_ids)]


##======== IDENTIFY FALSE POSITIVE MGIT SAMPLES ===============
# Based on: https://www.nejm.org/doi/suppl/10.1056/NEJMoa1901814/suppl_file/nejmoa1901814_protocol.pdf page 410-455
# AFB:  - performed as auramine staining at baseline for MTB detection (keep these)
#       - ZN staining for validation of MGIT results
#  1. All cultures that are positive (LJ or MGIT) are getting a ZN-smear from an isolate taken from that culture
#  2. If MGIT is positive but the validation ZN-smear is negative, the MGIT is considered as false positive
#  
## Get samples where the validation ZN-smears of MGIT tests were negative


mgit_samples_with_neg_afb=mb_subset_clean.loc[(mb_subset_clean['MBTESTCD']=='AFB')&\
                      (~mb_subset_clean['MBSPID'].isna())& # smear belongs to sample from MGIT tube
                      (mb_subset_clean['RESULT']=='negative'),'SAMPLE_REFID'].unique().tolist()
                

## Extract the results of the matching culture - validation ZN smears
a=(mb_subset_clean[mb_subset_clean['SAMPLE_REFID'].isin(mgit_samples_with_neg_afb)].groupby(['SAMPLE_REFID','SPDEVID','MBGRPID','MBTESTCD','MBTSTDTL'],dropna=False).apply(lambda x:x['RESULT'])).reset_index()


## Function for checking samples where the MGIT and validating ZN-smear are discordant
def get_false_pos_ttp_samples(x):
    try:
        afb_res=x.loc[(x['MBTESTCD']=='AFB')\
                    &(x['MBTSTDTL']=='Identification'),'RESULT'].values[0]
        try:
            ttp_res=x.loc[x['MBTESTCD']=='MTB','RESULT'].values[0]
            
            if len(x.loc[x['MBTESTCD']=='MTB'])==0:
                print(x['SAMPLE_REFID'].unique())
                pass
            else:
                if afb_res=='negative' and ttp_res=='positive':
                    return 'false positive'

        ## If IndexError: the MGIT was negative, but a ZN smear was performed nonetheless
        except IndexError:
            pass     
    except IndexError:
            pass     

## Get possible false positives (manual page 454) and drop them, as these samples 
#  (defined by MBGRPID, as multiple samples can be taken from a sputum) have a negative AFB with a positive MGIT
false_pos_mgit_sample_idx=a.groupby(['SAMPLE_REFID','MBGRPID'],dropna=False).apply(get_false_pos_ttp_samples).reset_index()['MBGRPID'].tolist()
  
mb_subset_clean=mb_subset_clean[~mb_subset_clean['MBGRPID'].isin(false_pos_mgit_sample_idx)]

##======== DROP VALIDATION ZN-SMEARS OF CULTURE ISOLATES ===============
## These ZN-smears serve to check if the culture is false positive. 
#  (true for MGIT, for LJ not sure, as it is not specified in protocol)'
#  Because the information of these samples' positivity is already contained in the LJ or MGIT test result, we can drop these results.                                  

mb_subset_clean=mb_subset_clean.loc[~((mb_subset_clean['MBTESTCD']=='AFB')&\
                                    (mb_subset_clean['MBTSTDTL']=='Identification')),:]




##======== MERGE PARALLEL LJ RESULTS REPORTED FOR EACH DAY ===============
## TWO RESULTS OF LJ CULTURE ARE REPORTED FOR EACH SAMPLE: 
#  - AS CULTURE GROWTH (POSITIVE/NEGATIVE) AND  AS CATEGORICAL (NEG/1/2/3/4+)
#  - CALCULATE THE MODE OF THE LJ RESULTS PER DAY. 
#  - A LOW NUMBER OF PATIENTS HAVE 2 SAMPLES TAKEN ON THE SAME DAY ==>TREAT THEM AS PARALLEL SAMPLES AND TAKE THE MODE
#    OVER ALL SAMPLES TAKEN ON A GIVEN DAY
#  - IF THE POSITVE/NEGATIVES ARE TIED, TAKE THE RESULT AS POSITIVE

def get_mode_of_result(x):
    if len(x['STD_RESULT'].mode())==1:
        return x['STD_RESULT'].mode()[0]
        
    if len(x['STD_RESULT'].mode())==2:
        return 'positive'

    if len(x['STD_RESULT'].mode())>2:
        raise ValueError('Invalid mb result! (not "negative" or "positive")')  

a=(mb_subset_clean.groupby(['USUBJID','MBDY_estimated', 'STUDYID','MBTESTCD','MEDIATYP','MBTSTDTL','MBMETHOD'],dropna=False,group_keys=True).apply(lambda x: x[['STD_NUM_RESULT','STD_CAT_RESULT','STD_NUM_UNITS','STD_RESULT','SAMPLE_REFID']]))
b=a.reset_index()
merged_result=b.groupby(['USUBJID','MBDY_estimated','STUDYID','MBTESTCD','MBTSTDTL','MBMETHOD'],dropna=False).apply(get_mode_of_result)
merged_result=merged_result.reset_index()
merged_result=merged_result.rename(columns={0:'STD_RESULT'})
#merged_result['MBMETHOD']=np.nan
#merged_result.loc[merged_result['MBTESTCD']=='MTB','MBMETHOD']='MICROBIAL CULTURE, LIQUID'    



###=========== STANDARDISE THE NAMES OF AFB, LJ & MGIT CULTURES 
tb_18_std_res=merged_result.copy()

tb_18_std_res['STD_TEST']=np.nan
tb_18_std_res.loc[(tb_18_std_res['MBTESTCD']=='AFB')\
                  &(tb_18_std_res['MBMETHOD'].str.contains('ACID FAST')),'STD_TEST']='ZN-smear'

tb_18_std_res.loc[(tb_18_std_res['MBTESTCD']=='AFB')\
                  &(tb_18_std_res['MBMETHOD'].str.contains('AURAMINE')),'STD_TEST']='Auramine-smear'


tb_18_std_res.loc[(tb_18_std_res['MBTESTCD']=='MTBCMPLX')\
                  &(tb_18_std_res['MBMETHOD'].isna()),'STD_TEST']='MTB-complex'

tb_18_std_res.loc[(tb_18_std_res['MBTESTCD']=='MTBCMPLX')\
                  &(tb_18_std_res['MBMETHOD'].str.contains('LINE')),'STD_TEST']='HAIN-test'

tb_18_std_res.loc[(tb_18_std_res['MBTESTCD']=='MTBCMPLX')\
                  &(tb_18_std_res['MBMETHOD'].str.contains('IMMUNOCHROMATOGRAPHY')),'STD_TEST']='MPT64-Antigen-Test'

tb_18_std_res.loc[(tb_18_std_res['MBTESTCD']=='MTBCMPLX')\
                  &(tb_18_std_res['MBMETHOD'].str.contains('POLYMERASE')),'STD_TEST']='RT-PCR'


tb_18_std_res.loc[(tb_18_std_res['MBTESTCD']=='MTB')\
                  &(tb_18_std_res['MBMETHOD'].str.contains('LIQUID')),'STD_TEST']='MGIT'    

Num of initial samples 6729
6729
Num of samples after dropping contaminated MGIT + blood agar samples 5644


/tmp/ipykernel_184321/4272176378.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mb_subset.groupby(['STUDYID','MBTESTCD','MEDIATYP'],dropna=False).apply(lambda x: (x['MBTSTDTL'].value_counts()))#.reset_index()
/tmp/ipykernel_184321/4272176378.py:102: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  a=(mb_subset_clean[mb_subset_clean['SAMPLE_REFID'].isin(mgit_samples_with_neg_afb)].groupby(['SAMPLE_REFID','SPDEVID','

In [ ]:
merged_result['MBMETHOD'].value_counts(dropna=False)

In [41]:
tb_18_std_res

,USUBJID,MBDY_estimated,STUDYID,MBTESTCD,MBTSTDTL,MBMETHOD,STD_RESULT,STD_TEST
0,TB-1018/01-9002,-9.0,TB-1018,AFB,Categorical Count,ACID FAST STAIN,negative,ZN-smear
1,TB-1018/01-9002,1.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT
2,TB-1018/01-9002,7.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT
3,TB-1018/01-9002,14.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT
4,TB-1018/01-9002,28.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",positive,MGIT
...,...,...,...,...,...,...,...,...
2788,TB-1018/04-9014,218.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT
2789,TB-1018/04-9014,279.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT
2790,TB-1018/04-9014,352.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT
2791,TB-1018/04-9014,453.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT


In [22]:
tb_18_std_res#['STD_TEST'].value_counts(dropna=False)
#mb_subset['STD_NUM_RESULT'].dropna()

,USUBJID,MBDY_estimated,STUDYID,MBTESTCD,MBTSTDTL,MBMETHOD,STD_RESULT,STD_TEST
0,TB-1018/01-9002,-9.0,TB-1018,AFB,Categorical Count,ACID FAST STAIN,negative,ZN-smear
1,TB-1018/01-9002,1.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT
2,TB-1018/01-9002,7.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT
3,TB-1018/01-9002,14.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT
4,TB-1018/01-9002,28.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",positive,MGIT
...,...,...,...,...,...,...,...,...
2788,TB-1018/04-9014,218.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT
2789,TB-1018/04-9014,279.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT
2790,TB-1018/04-9014,352.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT
2791,TB-1018/04-9014,453.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT


# CONCATENATE STANDARDISED DATAFRAMES OF ALL STUDIES

In [23]:
mb_std_res=pd.concat([tb_18_std_res,tb_20_std_res,tb_21_std_res,tb_22_std_res],axis=0)
mb_std_res=mb_std_res.rename(columns={'STD_TEST':'STD_MBTEST'})
#mb_std_res['STD_NUM_RESULT']=np.nan
mb_std_res['USUBJID'] = mb_std_res['USUBJID'].str.replace('\\', '/', regex=False)

mb_std_res
mb_std_res.to_csv('../data/out_mb_wo_false_positives.csv.gz',compression='gzip')

In [12]:
mb_std_res=pd.read_csv('../data/out_mb_wo_false_positives.csv.gz',index_col=0)

/tmp/ipykernel_184321/4168125655.py:1: DtypeWarning: Columns (13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  mb_std_res=pd.read_csv('../data/out_mb_wo_false_positives.csv.gz',index_col=0)


In [27]:
mb_std_res[mb_std_res['STUDYID']=='TB-1020']['STD_MBTEST'].unique()

array(['ZN-smear', 'MGIT', 'MTB-complex'], dtype=object)

In [ ]:
mb_std_res.columns
#mb[mb['USUBJID']=='TB-1022/21396']
mb_std_res.loc[(mb_std_res['STUDYID']=='TB-1021')&(mb_std_res['STD_MBTEST']=='MGIT'),:]#'STD_NUM_UNITS']

In [25]:
mb_std_res[mb_std_res['USUBJID']=='TB-1020/1006']
#mb_std_res[mb_std_res['STUDYID']=='TB-1022']['MBDY_estimated'].hist(bins=100,range=(-10,300))

,USUBJID,MBDY_estimated,STUDYID,MBTESTCD,MBTSTDTL,MBMETHOD,STD_RESULT,STD_MBTEST,STD_CAT_ORDINAL_RESULT,STD_CAT_RESULT,STD_NUM_RESULT,CULTURE_STATUS,STD_NUM_UNIT,MEDIATYP
62,TB-1020/1006,1.0,TB-1020,NaN,NaN,NaN,positive,ZN-smear,2.0,2+,NaN,NaN,NaN,NaN
63,TB-1020/1006,57.0,TB-1020,NaN,NaN,NaN,negative,ZN-smear,0.0,NEGATIVE,NaN,NaN,NaN,NaN
64,TB-1020/1006,85.0,TB-1020,NaN,NaN,NaN,negative,ZN-smear,0.0,NEGATIVE,NaN,NaN,NaN,NaN
65,TB-1020/1006,118.0,TB-1020,NaN,NaN,NaN,negative,ZN-smear,0.0,NEGATIVE,NaN,NaN,NaN,NaN
66,TB-1020/1006,148.0,TB-1020,NaN,NaN,NaN,negative,ZN-smear,0.0,NEGATIVE,NaN,NaN,NaN,NaN
67,TB-1020/1006,180.0,TB-1020,NaN,NaN,NaN,negative,ZN-smear,0.0,NEGATIVE,NaN,NaN,NaN,NaN
68,TB-1020/1006,195.0,TB-1020,NaN,NaN,NaN,negative,ZN-smear,0.0,NEGATIVE,NaN,NaN,NaN,NaN
69,TB-1020/1006,216.0,TB-1020,NaN,NaN,NaN,negative,ZN-smear,0.0,NEGATIVE,NaN,NaN,NaN,NaN
70,TB-1020/1006,244.0,TB-1020,NaN,NaN,NaN,negative,ZN-smear,0.0,NEGATIVE,NaN,NaN,NaN,NaN
71,TB-1020/1006,272.0,TB-1020,NaN,NaN,NaN,negative,ZN-smear,0.0,NEGATIVE,NaN,NaN,NaN,NaN


In [ ]:
a = mb_std_res[mb_std_res['STUDYID']=='TB-1021'].dropna(how='all',axis=0)

In [ ]:
a[a['STD_MBTEST']=='LJ-culture']